# Descarga de ensamblados → Drive

Verifica los ensamblados candidatos contra NCBI y baja los confirmados a
`tesis/70_genomas/`, sin pasar por tu disco.

**Por qué acá:** la sesión de Claude tiene NCBI bloqueado por política, así que
no puede ni verificar ni bajar. Colab sí.

Toda la lógica vive en `scripts/fetch_genomes.sh` — este notebook solo lo
maneja. Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def clonar():
    if CLON.exists():
        r = subprocess.run(['git', '-C', str(CLON), 'pull', '--ff-only'],
                           capture_output=True, text=True)
        return 'ya estaba clonado; ' + (r.stdout.strip() or r.stderr.strip())

    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
import glob, os, subprocess, shutil

# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Por eso esta celda esta en los cuatro y es idempotente: si las
# herramientas ya estan, no hace nada.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


def instala_sra():
    # ya desempaquetado en esta VM de una corrida anterior de la celda
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if c:
        _en_path(c[0]); 
    if shutil.which('prefetch') and shutil.which('vdb-validate'):
        return 'ya estaba'

    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode == 0 and sh('tar -xzf /tmp/sra.tar.gz -C /opt').returncode == 0:
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
        if c:
            _en_path(c[0])
            return f'tarball oficial {SRA_VER}'

    # La version del tarball puede cambiar o desaparecer. apt es mas viejo, pero
    # aca solo se descarga y se valida: nada de esto entra en la tesis.
    if sh('apt-get -qq install -y sra-toolkit').returncode == 0 and shutil.which('prefetch'):
        return 'apt (version distinta del tarball)'

    raise RuntimeError(
        'No pude instalar sra-tools ni por tarball ni por apt. '
        'Revisa la version vigente en https://github.com/ncbi/sra-tools/wiki '
        'y ajusta SRA_VER.')


if not shutil.which('jq'):
    sh('apt-get -qq update'); sh('apt-get -qq install -y jq')
print('sra-tools:', instala_sra())

faltan = [b for b in ('prefetch', 'vdb-validate', 'jq', 'curl', 'git')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('faltan herramientas: ' + ', '.join(faltan))
print('herramientas OK:', 'prefetch vdb-validate jq curl git')

In [ ]:
GENOMAS = DRIVE / '70_genomas'
GENOMAS.mkdir(parents=True, exist_ok=True)
print('destino:', GENOMAS)

## 1. Qué falta

`candidato` = propuesto pero **no comprobado**. El script se niega a bajar esos
hasta que una persona los verifique. Un ensamblado equivocado no falla
ruidosamente: alinea peor y contamina la anotación.

In [ ]:
!cd /content/tesis && ./scripts/fetch_genomes.sh estado

## 2. Verificar contra NCBI

Para cada organismo pregunta dos cosas: si el accession propuesto existe, y cuál
es el ensamblado de **referencia vigente** de la especie. La segunda importa más
que la primera — un accession puede existir y no ser el que corresponde.

**Leé la salida antes de seguir.**

In [ ]:
!cd /content/tesis && ./scripts/fetch_genomes.sh resolve

## 3. Confirmar

Poné `verificado` en `data/genomas.tsv` para los que la salida de arriba
confirmó. Editá la lista y corré la celda.

Si el vigente **difiere** del candidato, actualizá también el accession: gana
NCBI, no lo que dice el TSV.

In [ ]:
CONFIRMADOS = []          # p.ej. ['prupe', 'gadmo']
CORREGIR = {}             # p.ej. {'galga': 'GCF_016699485.2'}

import re, pathlib
spec = pathlib.Path('/content/tesis') / 'data' / 'genomas.tsv'
lineas = spec.read_text().split('\n')
for i, ln in enumerate(lineas):
    if ln.startswith('#') or '\t' not in ln:
        continue
    f = ln.split('\t')
    if f[0] in CORREGIR:
        f[4] = CORREGIR[f[0]]
    if f[0] in CONFIRMADOS:
        f[5] = 'verificado'
        lineas[i] = '\t'.join(f)
    elif f[0] in CORREGIR:
        lineas[i] = '\t'.join(f)
spec.write_text('\n'.join(lineas))
print('marcados:', CONFIRMADOS or '(ninguno — no se va a bajar nada)')

## 4. Bajar a Drive

In [ ]:
!cd /content/tesis && GENOMES_DIR=/content/drive/MyDrive/tesis/70_genomas ./scripts/fetch_genomes.sh fetch

## 5. Cerrar el círculo con git

El clon es efímero: se pierde al cerrar la sesión. Copiá esta salida al repo y
commiteala — **el checksum versionado es lo que deja constancia de qué genoma se
usó**, porque el que queda al lado del FASTA en Drive no prueba nada: quien
reemplace el genoma reemplaza el checksum con él.

In [ ]:
led = pathlib.Path('/content/tesis') / 'data' / 'genomas.sha256'
print('--- data/genomas.sha256 ---')
print(led.read_text() if led.exists() else '(vacío: no se bajó nada)')
print('--- data/genomas.tsv ---')
print(spec.read_text())